# ICCAD 2026 — Cross-Solver Benchmark (Parallel)

Runs the full ASP/clingo vs CP-SAT benchmark with **parallel execution**.
Each (testcase, profile) pair runs in its own process.

**Runtime**: ~2–3 min on Colab (vs 10+ min sequential on local).

In [ ]:
# Install dependencies
!pip install -q clingo ortools

In [ ]:
import os, json, re, time, textwrap
from pathlib import Path
from multiprocessing import Pool, cpu_count
from dataclasses import dataclass, field

print(f"CPUs available: {cpu_count()}")

BASE = Path("/content/iccad_bench")
CLINGO_DIR = BASE / "Clingo"
TESTCASE_DIR = BASE / "testCases"
CLINGO_DIR.mkdir(parents=True, exist_ok=True)
TESTCASE_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# ============================================================
# Write all .lp files to disk
# ============================================================

FILES = {}

# --- Catalog ---
FILES["Clingo/security_features_inst.lp"] = textwrap.dedent("""\
security_feature(zero_trust).
security_feature(mac).
security_feature(dynamic_mac).
logging_feature(zero_trust_logger).
logging_feature(some_logging).
logging_feature(no_logging).

power_cost(zero_trust, byAsset, 2).
power_cost(zero_trust, byComponent, 3).
power_cost(zero_trust, base, 3).
power_cost(mac, byAsset, 1).
power_cost(mac, byComponent, 2).
power_cost(mac, base, 0).
power_cost(dynamic_mac, byAsset, 1).
power_cost(dynamic_mac, byComponent, 2).
power_cost(dynamic_mac, base, 2).
power_cost(zero_trust_logger, base, 717).
power_cost(some_logging, base, 72).
power_cost(no_logging, base, 0).

vulnerability(zero_trust, 10).
vulnerability(mac, 30).
vulnerability(dynamic_mac, 20).
logging(zero_trust_logger, 5).
logging(some_logging, 10).
logging(no_logging, 20).

luts(zero_trust, byAsset, 90).
luts(zero_trust, byComponent, 380).
luts(zero_trust, base, 3200).
luts(mac, byAsset, 51).
luts(mac, byComponent, 234).
luts(mac, base, 0).
luts(dynamic_mac, byAsset, 51).
luts(dynamic_mac, byComponent, 234).
luts(dynamic_mac, base, 1985).
luts(zero_trust_logger, base, 31354).
luts(some_logging, base, 3135).
luts(no_logging, base, 0).

ffs(zero_trust, byAsset, 56).
ffs(zero_trust, byComponent, 480).
ffs(zero_trust, base, 2600).
ffs(mac, byAsset, 32).
ffs(mac, byComponent, 292).
ffs(mac, base, 0).
ffs(dynamic_mac, byAsset, 32).
ffs(dynamic_mac, byComponent, 292).
ffs(dynamic_mac, base, 1612).
ffs(zero_trust_logger, base, 25020).
ffs(some_logging, base, 2502).
ffs(no_logging, base, 0).

dsps(zero_trust, byAsset, 0). dsps(zero_trust, byComponent, 0). dsps(zero_trust, base, 0).
dsps(mac, byAsset, 0). dsps(mac, byComponent, 0). dsps(mac, base, 0).
dsps(dynamic_mac, byAsset, 0). dsps(dynamic_mac, byComponent, 0). dsps(dynamic_mac, base, 0).
dsps(zero_trust_logger, base, 220). dsps(some_logging, base, 22). dsps(no_logging, base, 0).

lutram(zero_trust, byAsset, 0). lutram(zero_trust, byComponent, 0). lutram(zero_trust, base, 201).
lutram(mac, byAsset, 0). lutram(mac, byComponent, 0). lutram(mac, base, 0).
lutram(dynamic_mac, byAsset, 0). lutram(dynamic_mac, byComponent, 0). lutram(dynamic_mac, base, 201).
lutram(zero_trust_logger, base, 154). lutram(some_logging, base, 2). lutram(no_logging, base, 0).

bram(zero_trust, byAsset, 0). bram(zero_trust, byComponent, 0). bram(zero_trust, base, 12).
bram(mac, byAsset, 0). bram(mac, byComponent, 0). bram(mac, base, 0).
bram(dynamic_mac, byAsset, 0). bram(dynamic_mac, byComponent, 0). bram(dynamic_mac, base, 8).
bram(zero_trust_logger, base, 18). bram(some_logging, base, 2). bram(no_logging, base, 0).

bufg(zero_trust, byAsset, 0). bufg(zero_trust, byComponent, 0). bufg(zero_trust, base, 0).
bufg(mac, byAsset, 0). bufg(mac, byComponent, 0). bufg(mac, base, 0).
bufg(dynamic_mac, byAsset, 0). bufg(dynamic_mac, byComponent, 0). bufg(dynamic_mac, base, 0).
bufg(zero_trust_logger, base, 0). bufg(some_logging, base, 0). bufg(no_logging, base, 0).

latency_cost(zero_trust, 8).
latency_cost(mac, 3).
latency_cost(dynamic_mac, 7).
latency_cost(zero_trust_logger, 22).
latency_cost(some_logging, 4).
latency_cost(no_logging, 0).
""")

# --- Encodings ---
FILES["Clingo/init_enc.lp"] = textwrap.dedent("""\
1 { selected_logging(C, L) : logging_feature(L) } 1 :- asset(C, _,_).
1 { selected_security(C, F) : security_feature(F) } 1 :- asset(C, _,_).
#show selected_logging/2.
#show selected_security/2.
""")

FILES["Clingo/opt_latency_enc.lp"] = textwrap.dedent("""\
asset_latency(A, Action, TotalLatency) :-
    asset(_, A, Action),
    selected_security(A, Security),
    latency_cost(Security, SecurityLatency),
    TotalLatency = SecurityLatency.
#show asset_latency/3.
""")

# opt_power_enc.lp and opt_resource_enc.lp are long; write from string
print("Writing .lp files...")

In [ ]:
# --- opt_power_enc.lp ---
FILES["Clingo/opt_power_enc.lp"] = textwrap.dedent("""\
security_feature_used(F) :- selected_security(C, F), component(C).
logging_feature_used(L) :- selected_logging(C, L), component(C).
asset_count(C, Count) :- component(C), Count = #count { A : asset(C, A, _) }.
component_power(C, TotalPower) :- component(C), SecurityComponentPower = #sum { Power : selected_security(C, F), power_cost(F, byComponent, Power) }, TotalPower = SecurityComponentPower.
#show component_power/2.
total_component_power(ComponentPower) :- ComponentPower = #sum { Power, C : component_power(C, Power) }.
#show total_component_power/1.
asset_power_for_component(C, TotalPower) :- component(C), TotalPower = #sum { Value, F : selected_security(C, F), asset_count(C, Count), power_cost(F, byAsset, Power), Value = Count * Power }.
total_asset_power(AssetPower) :- AssetPower = #sum { Power, C : asset_power_for_component(C, Power) }.
#show total_asset_power/1.
security_base_power(F, Power) :- security_feature_used(F), power_cost(F, base, Power).
logging_base_power(L, Power) :- logging_feature_used(L), power_cost(L, base, Power).
total_base_power(BasePower) :- SecurityBasePower = #sum { Power, F : security_base_power(F, Power) }, LoggingBasePower = #sum { Power, L : logging_base_power(L, Power) }, BasePower = SecurityBasePower + LoggingBasePower.
#show total_base_power/1.
total_power_used(TotalUsedPower) :- total_asset_power(TotalAssetPower), total_component_power(ComponentPower), total_base_power(BasePower), TotalUsedPower = TotalAssetPower + ComponentPower + BasePower.
#show total_power_used/1.
:- total_power_used(TotalUsedPower), system_capability(max_power, MaxPower), TotalUsedPower > MaxPower.
""")

# --- opt_resource_enc.lp (LUTs, FFs, DSPs, LUTRAM, BUFG, BRAM) ---
# Generate from template for each resource
_RES_TEMPLATE = """
{r}_used_by_component(C, Total) :- component(C), Total = #sum {{ Val : selected_security(C, F), {r}(F, byComponent, Val) }}.
total_component_{r}(S) :- S = #sum {{ Val, C : {r}_used_by_component(C, Val) }}.
asset_{r}_for_component(C, Total) :- component(C), Total = #sum {{ Value, F : selected_security(C, F), asset_count(C, Count), {r}(F, byAsset, V), Value = Count * V }}.
total_asset_{r}(S) :- S = #sum {{ Val, C : asset_{r}_for_component(C, Val) }}.
security_base_{r}(F, V) :- security_feature_used(F), {r}(F, base, V).
logging_base_{r}(L, V) :- logging_feature_used(L), {r}(L, base, V).
total_base_{r}(B) :- SB = #sum {{ V, F : security_base_{r}(F, V) }}, LB = #sum {{ V, L : logging_base_{r}(L, V) }}, B = SB + LB.
total_{r}_used(T) :- total_asset_{r}(A), total_component_{r}(C), total_base_{r}(B), T = A + C + B.
#show total_{r}_used/1.
:- total_{r}_used(T), system_capability(max_{r}, M), T > M.
"""

res_enc = "security_feature_used(F) :- selected_security(C, F), component(C).\n"
res_enc += "logging_feature_used(L) :- selected_logging(C, L), component(C).\n"
res_enc += "asset_count(C, Count) :- component(C), Count = #count { A : asset(C, A, _) }.\n"
for r in ["luts", "ffs", "dsps", "lutram", "bufg", "bram"]:
    res_enc += _RES_TEMPLATE.format(r=r)

FILES["Clingo/opt_resource_enc.lp"] = res_enc

In [ ]:
# --- opt_redundancy_generic_enc.lp ---
FILES["Clingo/opt_redundancy_generic_enc.lp"] = textwrap.dedent("""\
mu(25).
omega(1000).
#show impact/3.
#show asset/3.
vulnerability_score(C, V) :- selected_security(C, S), vulnerability(S, V).
logging_score(C, L)        :- selected_logging(C, Lf), logging(Lf, L).
#show vulnerability_score/2.
#show logging_score/2.
original_register_risk(C, R, Action, Risk) :- component(C), asset(C, R, Action), impact(R, Action, Imp), vulnerability_score(C, V), logging_score(C, L), Risk = Imp * V * L / 10.
original_prob(C, Prob) :- vulnerability_score(C, V), logging_score(C, L), Prob = V * L.
original_prob_normalized(C, Norm) :- component(C), original_prob(C, P), mu(Mu), omega(Omega), Norm = (P - Mu) * 1000 / (Omega - Mu).
in_any_group(C) :- redundant_group(_, C).
group_size(G, N) :- redundant_group(G, _), N = #count { C : redundant_group(G, C) }.
group_rank(G, C, R) :- redundant_group(G, C), N = #count { C2 : redundant_group(G, C2), C2 < C }, R = N + 1.
partial_norm(G, 1, C, P) :- group_rank(G, C, 1), original_prob_normalized(C, P).
partial_norm(G, R, C, Norm) :- R > 1, group_rank(G, C, R), original_prob_normalized(C, Pcur), Rprev = R - 1, partial_norm(G, Rprev, _, Pprev), Norm = Pprev * Pcur / 1000.
combined_prob_norm(G, Norm) :- group_size(G, N), partial_norm(G, N, _, Norm).
new_prob_normalized(C, P) :- redundant_group(G, C), combined_prob_norm(G, P).
new_prob_denormalized(C, Denorm) :- new_prob_normalized(C, P), mu(Mu), omega(Omega), Denorm = (P * (Omega - Mu) / 1000) + Mu * 10.
new_risk(C, R, A, Risk) :- component(C), not in_any_group(C), original_register_risk(C, R, A, Risk).
new_risk(C, R, Action, Risk) :- component(C), asset(C, R, Action), impact(R, Action, Imp), new_prob_denormalized(C, P), Risk = Imp * P / 100.
:- new_risk(_, _, _, Risk), system_capability(max_asset_risk, MaxRisk), Risk > MaxRisk.
#minimize { Risk, C, Asset, Action : new_risk(C, Asset, Action, Risk) }.
#show original_register_risk/4.
#show original_prob/2.
#show original_prob_normalized/2.
#show new_prob_normalized/2.
#show new_prob_denormalized/2.
#show new_risk/4.
#show combined_prob_norm/2.
""")

In [ ]:
# --- Profiles ---
FILES["Clingo/tgt_system_inst1.lp"] = "system_capability(max_power,15000). system_capability(max_luts,53200). system_capability(max_ffs,106400). system_capability(max_dsps,220). system_capability(max_lutram,17400). system_capability(max_bufgs,32). system_capability(max_bram,140). system_capability(max_asset_risk,100).\n"
FILES["Clingo/tgt_system_inst2.lp"] = "system_capability(max_power,15000). system_capability(max_luts,10000). system_capability(max_ffs,106400). system_capability(max_dsps,220). system_capability(max_lutram,17400). system_capability(max_bufgs,32). system_capability(max_bram,140). system_capability(max_asset_risk,500).\n"
FILES["Clingo/tgt_system_inst3.lp"] = "system_capability(max_power,1000). system_capability(max_luts,53200). system_capability(max_ffs,106400). system_capability(max_dsps,220). system_capability(max_lutram,17400). system_capability(max_bufgs,32). system_capability(max_bram,140). system_capability(max_asset_risk,100).\n"

FILES["Clingo/tgt_system_inst_ot1.lp"] = "system_capability(max_power,15000). system_capability(max_luts,254200). system_capability(max_ffs,508400). system_capability(max_dsps,1540). system_capability(max_lutram,63550). system_capability(max_bufgs,32). system_capability(max_bram,795). system_capability(max_asset_risk,100).\n"
FILES["Clingo/tgt_system_inst_ot2.lp"] = "system_capability(max_power,15000). system_capability(max_luts,25000). system_capability(max_ffs,508400). system_capability(max_dsps,1540). system_capability(max_lutram,63550). system_capability(max_bufgs,32). system_capability(max_bram,795). system_capability(max_asset_risk,500).\n"
FILES["Clingo/tgt_system_inst_ot3.lp"] = "system_capability(max_power,800). system_capability(max_luts,254200). system_capability(max_ffs,508400). system_capability(max_dsps,1540). system_capability(max_lutram,63550). system_capability(max_bufgs,32). system_capability(max_bram,795). system_capability(max_asset_risk,100).\n"

In [ ]:
# --- Test cases (synthetic) ---
# TC1: 3 components, no groups
FILES["testCases/testCase1_inst.lp"] = textwrap.dedent("""\
component(p1; c1; c2).
asset(p1, p1_r1, read). asset(p1, p1_r1, write).
asset(c1, c1_r1, read). asset(c1, c1_r1, write).
asset(c2, c2_r1, read). asset(c2, c2_r1, write).
impact(p1_r1, read, 5). impact(p1_r1, write, 5).
impact(c1_r1, read, 3). impact(c1_r1, write, 3).
impact(c2_r1, read, 1). impact(c2_r1, write, 1).
allowable_latency(p1_r1, read, 10). allowable_latency(p1_r1, write, 10).
allowable_latency(c1_r1, read, 10). allowable_latency(c1_r1, write, 10).
allowable_latency(c2_r1, read, 10). allowable_latency(c2_r1, write, 10).
""")

# TC2-TC9: read from local repo or define inline
# For brevity, we'll read them from uploaded files or skip synthetics
# and focus on OT. Upload testCases/*.lp files if you want full suite.
print("Synthetic test cases: upload testCases/*.lp for full suite, or run OT-only below.")

In [ ]:
# --- OpenTitan test case ---
FILES["testCases/testCaseOT_inst.lp"] = textwrap.dedent("""\
component(cpu; dma; aes; hmac; kmac; otbn; keymgr; otp; lc; flash; sram; rom; uart0; uart1; gpio; spi; i2c; timer; alert; entropy).
asset(cpu, cpu_r1, read). asset(cpu, cpu_r1, write).
asset(dma, dma_r1, read). asset(dma, dma_r1, write).
asset(aes, aes_r1, read). asset(aes, aes_r1, write).
asset(hmac, hmac_r1, read). asset(hmac, hmac_r1, write).
asset(kmac, kmac_r1, read). asset(kmac, kmac_r1, write).
asset(otbn, otbn_r1, read). asset(otbn, otbn_r1, write).
asset(keymgr, keymgr_r1, read). asset(keymgr, keymgr_r1, write).
asset(otp, otp_r1, read). asset(otp, otp_r1, write).
asset(lc, lc_r1, read). asset(lc, lc_r1, write).
asset(flash, flash_r1, read). asset(flash, flash_r1, write).
asset(sram, sram_r1, read). asset(sram, sram_r1, write).
asset(rom, rom_r1, read). asset(rom, rom_r1, write).
asset(uart0, uart0_r1, read). asset(uart0, uart0_r1, write).
asset(uart1, uart1_r1, read). asset(uart1, uart1_r1, write).
asset(gpio, gpio_r1, read). asset(gpio, gpio_r1, write).
asset(spi, spi_r1, read). asset(spi, spi_r1, write).
asset(i2c, i2c_r1, read). asset(i2c, i2c_r1, write).
asset(timer, timer_r1, read). asset(timer, timer_r1, write).
asset(alert, alert_r1, read). asset(alert, alert_r1, write).
asset(entropy, entropy_r1, read). asset(entropy, entropy_r1, write).
redundant_group(1, aes). redundant_group(1, hmac). redundant_group(1, kmac).
redundant_group(2, uart0). redundant_group(2, uart1).
impact(cpu_r1, read, 10). impact(cpu_r1, write, 10).
impact(dma_r1, read, 8). impact(dma_r1, write, 8).
impact(aes_r1, read, 9). impact(aes_r1, write, 10).
impact(hmac_r1, read, 8). impact(hmac_r1, write, 9).
impact(kmac_r1, read, 8). impact(kmac_r1, write, 9).
impact(otbn_r1, read, 9). impact(otbn_r1, write, 10).
impact(keymgr_r1, read, 10). impact(keymgr_r1, write, 10).
impact(otp_r1, read, 9). impact(otp_r1, write, 9).
impact(lc_r1, read, 9). impact(lc_r1, write, 8).
impact(flash_r1, read, 7). impact(flash_r1, write, 8).
impact(sram_r1, read, 5). impact(sram_r1, write, 6).
impact(rom_r1, read, 6). impact(rom_r1, write, 3).
impact(uart0_r1, read, 3). impact(uart0_r1, write, 3).
impact(uart1_r1, read, 3). impact(uart1_r1, write, 3).
impact(gpio_r1, read, 2). impact(gpio_r1, write, 4).
impact(spi_r1, read, 4). impact(spi_r1, write, 5).
impact(i2c_r1, read, 3). impact(i2c_r1, write, 4).
impact(timer_r1, read, 2). impact(timer_r1, write, 2).
impact(alert_r1, read, 7). impact(alert_r1, write, 5).
impact(entropy_r1, read, 8). impact(entropy_r1, write, 7).
allowable_latency(cpu_r1, read, 5). allowable_latency(cpu_r1, write, 5).
allowable_latency(dma_r1, read, 5). allowable_latency(dma_r1, write, 5).
allowable_latency(aes_r1, read, 7). allowable_latency(aes_r1, write, 7).
allowable_latency(hmac_r1, read, 10). allowable_latency(hmac_r1, write, 10).
allowable_latency(kmac_r1, read, 10). allowable_latency(kmac_r1, write, 10).
allowable_latency(otbn_r1, read, 7). allowable_latency(otbn_r1, write, 7).
allowable_latency(keymgr_r1, read, 5). allowable_latency(keymgr_r1, write, 5).
allowable_latency(otp_r1, read, 10). allowable_latency(otp_r1, write, 10).
allowable_latency(lc_r1, read, 15). allowable_latency(lc_r1, write, 15).
allowable_latency(flash_r1, read, 15). allowable_latency(flash_r1, write, 20).
allowable_latency(sram_r1, read, 10). allowable_latency(sram_r1, write, 10).
allowable_latency(rom_r1, read, 10). allowable_latency(rom_r1, write, 1000).
allowable_latency(uart0_r1, read, 100). allowable_latency(uart0_r1, write, 100).
allowable_latency(uart1_r1, read, 100). allowable_latency(uart1_r1, write, 100).
allowable_latency(gpio_r1, read, 1000). allowable_latency(gpio_r1, write, 1000).
allowable_latency(spi_r1, read, 50). allowable_latency(spi_r1, write, 50).
allowable_latency(i2c_r1, read, 100). allowable_latency(i2c_r1, write, 100).
allowable_latency(timer_r1, read, 1000). allowable_latency(timer_r1, write, 1000).
allowable_latency(alert_r1, read, 10). allowable_latency(alert_r1, write, 15).
allowable_latency(entropy_r1, read, 10). allowable_latency(entropy_r1, write, 15).
""")

# --- Write all files to disk ---
for relpath, content in FILES.items():
    p = BASE / relpath
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(content)

print(f"Wrote {len(FILES)} files to {BASE}")

In [ ]:
# ============================================================
# Solver: ASP/clingo
# ============================================================

def solve_clingo(tc_file, profile_file, timeout=120.0):
    """Solve one (testcase, profile) with clingo. Returns dict."""
    import clingo

    lp_files = [
        str(CLINGO_DIR / "init_enc.lp"),
        str(CLINGO_DIR / "security_features_inst.lp"),
        str(CLINGO_DIR / "opt_redundancy_generic_enc.lp"),
        str(CLINGO_DIR / "opt_resource_enc.lp"),
        str(CLINGO_DIR / "opt_power_enc.lp"),
        str(CLINGO_DIR / "opt_latency_enc.lp"),
        str(tc_file),
        str(profile_file),
    ]

    last_atoms, last_cost = [], []

    def on_model(model):
        nonlocal last_atoms, last_cost
        last_atoms = list(model.symbols(shown=True))
        last_cost = list(model.cost)

    ctl = clingo.Control(["--opt-mode=opt", "0"])
    for f in lp_files:
        ctl.load(f)
    ctl.ground([("base", [])])

    t0 = time.perf_counter()
    with ctl.solve(on_model=on_model, async_=True) as handle:
        finished = handle.wait(timeout)
        if not finished:
            handle.cancel()
    elapsed = time.perf_counter() - t0

    result = {"time_s": round(elapsed, 3), "objective": 0, "optimal": False, "security": {}, "logging": {}, "luts": 0, "ffs": 0, "power": 0}
    if last_cost:
        result["objective"] = last_cost[0]
        result["optimal"] = finished
        for sym in last_atoms:
            a = sym.arguments
            if sym.name == "selected_security" and len(a) == 2:
                result["security"][str(a[0])] = str(a[1])
            elif sym.name == "selected_logging" and len(a) == 2:
                result["logging"][str(a[0])] = str(a[1])
            elif sym.name == "total_luts_used" and len(a) == 1:
                result["luts"] = a[0].number
            elif sym.name == "total_ffs_used" and len(a) == 1:
                result["ffs"] = a[0].number
            elif sym.name == "total_power_used" and len(a) == 1:
                result["power"] = a[0].number
    return result

In [ ]:
# ============================================================
# Solver: CP-SAT (simplified, self-contained)
# ============================================================
from ortools.sat.python import cp_model

def _parse_lp_facts(path):
    """Minimal .lp parser returning fact tuples."""
    text = Path(path).read_text()
    facts = []
    for m in re.finditer(r'(\w+)\(([^)]+)\)\.', text):
        name = m.group(1)
        args = [a.strip() for a in m.group(2).split(',')]
        facts.append((name, args))
    # Handle semicolons in component declarations
    for m in re.finditer(r'component\(([^)]+)\)\.', text):
        parts = [p.strip() for p in m.group(1).split(';')]
        if len(parts) > 1:
            for p in parts:
                facts.append(('component', [p]))
    return facts


def solve_cpsat(tc_file, profile_file, catalog_file):
    """Solve one (testcase, profile) with CP-SAT. Returns dict."""
    MU, OMEGA = 25, 1000
    cat_facts = _parse_lp_facts(catalog_file)
    tc_facts = _parse_lp_facts(tc_file)
    prof_facts = _parse_lp_facts(profile_file)

    # Extract catalog
    sec_features, log_features = [], []
    vuln, log_score, lat_cost = {}, {}, {}
    res_costs = {}  # (resource, feature, tier) -> int

    for name, args in cat_facts:
        if name == 'security_feature': sec_features.append(args[0])
        elif name == 'logging_feature': log_features.append(args[0])
        elif name == 'vulnerability': vuln[args[0]] = int(args[1])
        elif name == 'logging': log_score[args[0]] = int(args[1])
        elif name == 'latency_cost': lat_cost[args[0]] = int(args[1])
        elif name in ('luts','ffs','dsps','lutram','bram','bufg','power_cost'):
            rn = 'power' if name == 'power_cost' else name
            res_costs[(rn, args[0], args[1])] = int(args[2])

    # Extract test case
    components, assets, impact, groups = [], [], {}, {}
    allowable_lat = {}
    for name, args in tc_facts:
        if name == 'component' and len(args) == 1:
            if args[0] not in components: components.append(args[0])
        elif name == 'asset': assets.append((args[0], args[1], args[2]))
        elif name == 'impact': impact[(args[0], args[1])] = int(args[2])
        elif name == 'redundant_group': groups.setdefault(args[0], []).append(args[1])
        elif name == 'allowable_latency': allowable_lat[(args[0], args[1])] = int(args[2])

    # Extract profile
    caps = {}
    for name, args in prof_facts:
        if name == 'system_capability': caps[args[0]] = int(args[1])

    def _cost(res, feat, tier): return res_costs.get((res, feat, tier), 0)

    # Deduplicate components from semicolon parsing
    components = list(dict.fromkeys(components))

    n_sec, n_log = len(sec_features), len(log_features)
    n_pairs = n_sec * n_log

    # Asset count per component
    comp_n = {c: len({a for cc, a, _ in assets if cc == c}) for c in components}

    # V*L product table
    pair_vl = []
    pair_sec_name, pair_log_name = [], []
    for si, s in enumerate(sec_features):
        for li, l in enumerate(log_features):
            pair_sec_name.append(s)
            pair_log_name.append(l)
            pair_vl.append(vuln.get(s, 0) * log_score.get(l, 0))

    model = cp_model.CpModel()

    # Decision variables
    pv = {c: model.new_int_var(0, n_pairs - 1, f'p_{c}') for c in components}
    comp_si = {c: model.new_int_var(0, n_sec - 1, f'si_{c}') for c in components}
    comp_li = {c: model.new_int_var(0, n_log - 1, f'li_{c}') for c in components}
    comp_vl = {}
    for c in components:
        model.add_division_equality(comp_si[c], pv[c], n_log)
        model.add_modulo_equality(comp_li[c], pv[c], n_log)
        comp_vl[c] = model.new_int_var(min(pair_vl), max(pair_vl), f'vl_{c}')
        model.add_element(pv[c], pair_vl, comp_vl[c])

    # Resource constraints
    ALL_RES = ('luts', 'ffs', 'dsps', 'lutram', 'bram', 'bufg', 'power')
    comp_res = {r: {} for r in ALL_RES}
    for c in components:
        na = comp_n.get(c, 1)
        for r in ALL_RES:
            vals = []
            for si, s in enumerate(sec_features):
                cost = _cost(r, s, 'byComponent') + _cost(r, s, 'byAsset') * na
                for li, l in enumerate(log_features):
                    vals.append(cost)
            v = model.new_int_var(min(vals), max(vals), f'{r}_{c}')
            model.add_element(pv[c], vals, v)
            comp_res[r][c] = v

    # Base cost tracking
    sec_used = {}
    for si in range(n_sec):
        sec_used[si] = model.new_bool_var(f'su_{si}')
        inds = []
        for c in components:
            b = model.new_bool_var(f's{si}_{c}')
            model.add(comp_si[c] == si).only_enforce_if(b)
            model.add(comp_si[c] != si).only_enforce_if(b.negated())
            inds.append(b)
        model.add_max_equality(sec_used[si], inds)

    log_used = {}
    for li in range(n_log):
        log_used[li] = model.new_bool_var(f'lu_{li}')
        inds = []
        for c in components:
            b = model.new_bool_var(f'l{li}_{c}')
            model.add(comp_li[c] == li).only_enforce_if(b)
            model.add(comp_li[c] != li).only_enforce_if(b.negated())
            inds.append(b)
        model.add_max_equality(log_used[li], inds)

    for r in ALL_RES:
        cap_key = f'max_{r}'
        if r == 'bufg': cap_key = 'max_bufgs'
        if cap_key not in caps: continue
        comp_sum = sum(comp_res[r][c] for c in components)
        base_terms = []
        for si, s in enumerate(sec_features):
            bv = _cost(r, s, 'base')
            if bv: base_terms.append(bv * sec_used[si])
        for li, l in enumerate(log_features):
            bv = _cost(r, l, 'base')
            if bv: base_terms.append(bv * log_used[li])
        model.add(comp_sum + sum(base_terms) <= caps[cap_key])

    # --- Redundancy-aware risk ---
    in_group = set()
    for members in groups.values():
        for m in members: in_group.add(m)

    comp_norm = {}
    for c in in_group:
        lo_n = (min(pair_vl) - MU) * 1000
        hi_n = (max(pair_vl) - MU) * 1000
        num = model.new_int_var(lo_n, hi_n, f'nn_{c}')
        model.add(num == (comp_vl[c] - MU) * 1000)
        nlo = lo_n // (OMEGA - MU) if lo_n >= 0 else -((-lo_n + OMEGA - MU - 1) // (OMEGA - MU))
        norm = model.new_int_var(nlo, 1000, f'n_{c}')
        model.add_division_equality(norm, num, OMEGA - MU)
        comp_norm[c] = norm

    group_combined = {}
    for gid, members in groups.items():
        ranked = sorted(members)
        if len(ranked) == 1:
            group_combined[gid] = comp_norm[ranked[0]]
            continue
        prev = comp_norm[ranked[0]]
        for k in range(1, len(ranked)):
            pk = model.new_int_var(-1000*1000, 1000*1000, f'gp_{gid}_{k}')
            model.add_multiplication_equality(pk, [prev, comp_norm[ranked[k]]])
            nv = model.new_int_var(-1000, 1000, f'gn_{gid}_{k}')
            model.add_division_equality(nv, pk, 1000)
            prev = nv
        group_combined[gid] = prev

    comp_combined = {}
    for gid, members in groups.items():
        for m in members:
            comp_combined[m] = group_combined[gid]

    comp_denorm = {}
    for c in in_group:
        pd = model.new_int_var(-1000*(OMEGA-MU), 1000*(OMEGA-MU), f'dp_{c}')
        model.add(pd == comp_combined[c] * (OMEGA - MU))
        sc = model.new_int_var(-(OMEGA-MU), OMEGA-MU, f'ds_{c}')
        model.add_division_equality(sc, pd, 1000)
        dn = model.new_int_var(-(OMEGA-MU)+MU*10, (OMEGA-MU)+MU*10, f'dn_{c}')
        model.add(dn == sc + MU * 10)
        comp_denorm[c] = dn

    risk_cap = caps.get('max_asset_risk', 999999)
    max_vl = max(pair_vl)
    risk_vars = []
    for comp, asset_name, action in assets:
        imp = impact.get((asset_name, action), 0)
        if comp in in_group:
            rp = model.new_int_var(-imp*OMEGA*10, imp*OMEGA*10, f'rp_{comp}_{asset_name}_{action}')
            model.add(rp == imp * comp_denorm[comp])
            risk = model.new_int_var(-imp*OMEGA*10//100-1, imp*OMEGA*10//100+1, f'r_{comp}_{asset_name}_{action}')
            model.add_division_equality(risk, rp, 100)
        else:
            rp = model.new_int_var(0, imp*max_vl, f'rp_{comp}_{asset_name}_{action}')
            model.add(rp == imp * comp_vl[comp])
            risk = model.new_int_var(0, imp*max_vl//10+1, f'r_{comp}_{asset_name}_{action}')
            model.add_division_equality(risk, rp, 10)
        model.add(risk <= risk_cap)
        risk_vars.append(risk)

    model.minimize(sum(risk_vars))

    solver = cp_model.CpSolver()
    solver.parameters.max_time_in_seconds = 120
    solver.parameters.num_workers = 1

    t0 = time.perf_counter()
    status = solver.solve(model)
    elapsed = time.perf_counter() - t0

    result = {"time_s": round(elapsed, 3), "objective": 0, "optimal": False, "security": {}, "logging": {}, "luts": 0, "ffs": 0, "power": 0}
    if status in (cp_model.OPTIMAL, cp_model.FEASIBLE):
        result["objective"] = int(solver.objective_value)
        result["optimal"] = (status == cp_model.OPTIMAL)
        for c in components:
            idx = solver.value(pv[c])
            result["security"][c] = pair_sec_name[idx]
            result["logging"][c] = pair_log_name[idx]
        for rn, attr in [('luts','luts'),('ffs','ffs'),('power','power')]:
            ct = sum(solver.value(comp_res[rn][c]) for c in components)
            for si, s in enumerate(sec_features):
                if solver.value(sec_used[si]): ct += _cost(rn, s, 'base')
            for li, l in enumerate(log_features):
                if solver.value(log_used[li]): ct += _cost(rn, l, 'base')
            result[attr] = ct
    return result

In [ ]:
# ============================================================
# Parallel runner
# ============================================================

def _run_one(args):
    """Worker function for one (tc, profile) pair. Runs both solvers."""
    tc_name, tc_file, prof_label, prof_file, cat_file = args
    asp = solve_clingo(tc_file, prof_file)
    cpsat = solve_cpsat(tc_file, prof_file, cat_file)
    match = asp['objective'] == cpsat['objective']
    return {
        'testcase': tc_name, 'profile': prof_label,
        'asp_objective': asp['objective'], 'asp_time_s': asp['time_s'],
        'asp_luts': asp['luts'], 'asp_ffs': asp['ffs'], 'asp_power': asp['power'],
        'asp_security': asp['security'], 'asp_logging': asp['logging'],
        'cpsat_objective': cpsat['objective'], 'cpsat_time_s': cpsat['time_s'],
        'cpsat_luts': cpsat['luts'], 'cpsat_ffs': cpsat['ffs'], 'cpsat_power': cpsat['power'],
        'cpsat_security': cpsat['security'], 'cpsat_logging': cpsat['logging'],
        'objective_match': match,
    }


# Build job list
cat_file = str(CLINGO_DIR / 'security_features_inst.lp')

JOBS = []

# OT cases (the main bottleneck)
ot_profiles = [
    ('OT-A', 'tgt_system_inst_ot1.lp'),
    ('OT-B', 'tgt_system_inst_ot2.lp'),
    ('OT-C', 'tgt_system_inst_ot3.lp'),
]
for label, pf in ot_profiles:
    JOBS.append(('testCaseOT_inst', str(TESTCASE_DIR / 'testCaseOT_inst.lp'),
                 label, str(CLINGO_DIR / pf), cat_file))

# Synthetic cases (if files exist)
syn_profiles = [
    ('A', 'tgt_system_inst1.lp'),
    ('B', 'tgt_system_inst2.lp'),
    ('C', 'tgt_system_inst3.lp'),
]
for tc in ['testCase1_inst']:
    tc_path = TESTCASE_DIR / f'{tc}.lp'
    if tc_path.exists():
        for label, pf in syn_profiles:
            JOBS.append((tc, str(tc_path), label, str(CLINGO_DIR / pf), cat_file))

print(f'{len(JOBS)} jobs queued, running on {cpu_count()} CPUs...')
print()

t_start = time.perf_counter()
with Pool(processes=min(cpu_count(), len(JOBS))) as pool:
    results = pool.map(_run_one, JOBS)
t_total = time.perf_counter() - t_start

# Print results
print(f"{'Case':20s} {'Prof':>6s}  {'ASP':>6s} {'ASP(s)':>8s}  {'CPSAT':>6s} {'CPSAT(s)':>8s}  Match")
print('-' * 75)
for r in results:
    m = 'OK' if r['objective_match'] else 'MISMATCH'
    print(f"{r['testcase']:20s} {r['profile']:>6s}  {r['asp_objective']:>6d} {r['asp_time_s']:>8.3f}  {r['cpsat_objective']:>6d} {r['cpsat_time_s']:>8.3f}  {m}")

print(f"\nTotal wall-clock: {t_total:.1f}s (parallel)")
print(f"Sum ASP times:    {sum(r['asp_time_s'] for r in results):.1f}s")
print(f"Matches: {sum(1 for r in results if r['objective_match'])}/{len(results)}")

In [ ]:
# Save results JSON
out_path = '/content/iccad_cpsat_comparison.json'
with open(out_path, 'w') as f:
    json.dump(results, f, indent=2)
print(f'Saved {len(results)} results to {out_path}')

# Download link
try:
    from google.colab import files
    files.download(out_path)
except ImportError:
    pass